In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from redo import (
                create_transition_matrix, 
                simulate_c, 
                z_given_c_table, 
                simulate_z,
                simulate_x,
                simulate_hmm,
                plot_single,
                plot_freq,
                plot_all,
                plot_mean,
                make_datasets,
                x_given_z_vector,
                message_z_to_c,
                local_evidence_messages,
                forward_messages,
                backward_messages,
                posterior_c,
                posterior_z,
                hmm_pipeline,
                plot_c_posterior,
                plot_heatmap_z_posterior,
                plot_heatmap_z,
                lambda_hat_from_xz,
                alpha_hat_from_cz,
                beta_hat_from_C,
                gamma_hat_from_C,
                learn_all_params_from_known_data,
                hard_assigment_EM,
                init_lambda_kmeans,
                plot_convergence)

In [ ]:
# suggested values
alpha_val = 0.9
beta_val = 0.2
small_gamma_val = 0.1
lambda_0 = 1
lambda_1 = 5

n = 10
big_T = 100

In [ ]:
C, Z, X = simulate_hmm(big_T, n, alpha_val, beta_val, small_gamma_val, lambda_0, lambda_1)
plot_all(X)
plot_mean(X)

In [ ]:
x_features_data = []
y_labels_data = []

X_full_data = []
C_full_data = []
Z_full_data = []

t_indexes = [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]
for t_idx in t_indexes:
    # with seed = 123 + t_idx, we get a different dataset for each t_index
    x_features, y_labels, X_full, C_full, Z_full = make_datasets(500, t_idx, big_T, n, alpha_val, beta_val,
                                                                 small_gamma_val, lambda_0, lambda_1, seed=123 + t_idx)
    x_features_data.append(x_features)
    y_labels_data.append(y_labels)
    X_full_data.append(X_full)
    C_full_data.append(C_full)
    Z_full_data.append(Z_full)
print(x_features_data[0].shape) # feature shape: (M, n)
print(y_labels_data[0].shape)   # labels shape: (M,)
print("class counts:", np.bincount(y_labels_data[0], minlength=3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

x_train_data = []
x_test_data = []
y_train_data = []
y_test_data = []

X_full_train_data = []
X_full_test_data = []
C_full_train_data = []
C_full_test_data = []
Z_full_train_data = []
Z_full_test_data = []


for i in range(len(t_indexes)):
    x_train, x_test, y_train, y_test, X_full_train, X_full_test, C_full_train, C_full_test, Z_full_train, Z_full_test = train_test_split(
        x_features_data[i], y_labels_data[i], X_full_data[i], C_full_data[i], Z_full_data[i], test_size=0.2, random_state=123)

    x_train_data.append(x_train)
    x_test_data.append(x_test)
    y_train_data.append(y_train)
    y_test_data.append(y_test)

    X_full_train_data.append(X_full_train)
    X_full_test_data.append(X_full_test)
    C_full_train_data.append(C_full_train)
    C_full_test_data.append(C_full_test)
    Z_full_train_data.append(Z_full_train)
    Z_full_test_data.append(Z_full_test)

models = []
train_errors = []
test_errors = []

for i in range(len(t_indexes)):
    model = LogisticRegression(max_iter=5000, random_state=123)
    model.fit(x_train_data[i], y_train_data[i])
    
    train_errors.append(1 - model.score(x_train_data[i], y_train_data[i]))
    test_errors.append(1 - model.score(x_test_data[i], y_test_data[i]))
    models.append(model)

for i in range(len(t_indexes)):
    print(f"t={t_indexes[i]+1}: train_error={train_errors[i]:.6f}, test_error={test_errors[i]:.6f}")

In [ ]:
C_true, Z_true, X_sim = simulate_hmm(big_T, n, alpha_val, beta_val, small_gamma_val, lambda_0, lambda_1, seed=123)

res_sim = hmm_pipeline(X_sim, alpha_val, beta_val, small_gamma_val, lambda_0, lambda_1)

In [ ]:
print("C accuracy:", np.mean(res_sim["c_hat"] == C_true))
print("Z accuracy:", np.mean(res_sim["z_hat"] == Z_true))

plot_heatmap_z(Z_true, res_sim["qZ"])

In [ ]:
plot_c_posterior(res_sim)

In [ ]:
all_data = ["proj_HMM/Ex_1.csv", "proj_HMM/Ex_2.csv", "proj_HMM/Ex_3.csv", "proj_HMM/Ex_4.csv", "proj_HMM/Ex_5.csv", "proj_HMM/Ex_6.csv", "proj_HMM/Ex_7.csv", "proj_HMM/Ex_8.csv", "proj_HMM/Ex_9.csv", "proj_HMM/Ex_10.csv"]

read_data = []

for i in all_data:
    df = pd.read_csv(i)
    df.drop(columns=["t"], axis=1, inplace=True)
    df = df.to_numpy()
    read_data.append(df)

In [ ]:
for i in read_data:
    res = hmm_pipeline(i, alpha_val, beta_val, small_gamma_val, lambda_0, lambda_1)
    plot_c_posterior(res)
    plot_heatmap_z_posterior(res)

In [ ]:
est = learn_all_params_from_known_data(X_sim, C_true, Z_true)
est

In [ ]:
all_estimates = []

for seed in range(100):
    C_true, Z_true, X_sim = simulate_hmm(
        big_T,
        n,
        alpha_val,
        beta_val,
        small_gamma_val,
        lambda_0,
        lambda_1,
        seed
    )

    est = learn_all_params_from_known_data(X_sim, C_true, Z_true)
    all_estimates.append(est)

In [ ]:
keys = ["alpha_hat", "beta_hat", "gamma_hat", "lambda0_hat", "lambda1_hat"]

for key in keys:
    vals = np.array([d[key] for d in all_estimates])
    print(key, "mean = ", 
          vals.mean(), "std = ", 
          vals.std(), "min = ", 
          vals.min(), "max = ", 
          vals.max())

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(5, 12))

for ax, key in zip(axes, keys):
    vals = np.array([d[key] for d in all_estimates])
    ax.hist(vals)
    ax.set_title(key)
plt.tight_layout()
plt.show()

In [ ]:
C_true, Z_true, X_sim = simulate_hmm(big_T, n, alpha_val, beta_val, small_gamma_val, lambda_0, lambda_1, seed=123)

em_res = hard_assigment_EM(X_sim, 0.7, 0.4, 0.3 , 3.0, 4.0, 200)


print(em_res["alpha_hat"])
print(em_res["beta_hat"])
print(em_res["gamma_hat"])
print(em_res["lambda0_hat"])
print(em_res["lambda1_hat"])

In [ ]:

all_results = []

for i in range(len(all_data)):
    res = hard_assigment_EM(read_data[i])
    all_results.append(res)

In [ ]:
plot_convergence(em_res)

In [ ]:
for i in all_results:
    plot_convergence(i)